# nb23 - tSZ power spectrum of the unrotated (z = 0 - 0.35) Compton-y map

The map `data/map/y_unlensed_L2p8_m9_lc0_unrotated_z0_0p35_nside4096.fits` was
built from ONLY the seven unrotated lightcone shells (shells 0-6, identity
yang26 rotation), which tile comoving radius 3-1412 Mpc, i.e. **z = 0 to 0.35**
(equal dz = 0.05 bins).

We measure, exactly as before:
- the **full-sky** D_ell^yy (healpy `anafast`, lmax=2048, no pixel window),
- the **masked** D_ell^yy at q_theory > {5, 10, 20, 50} (5 R_500c discs at the
  detected clusters with z < 0.35, NaMaster C1 0.5 deg apodisation, decoupled
  pseudo-Cl),

and overlay the **hmfast** pure-gNFW theory with the bias fixed to the full-sky
best fit **B = 1.1093** (from `chains/mcmc_pureGNFW_fitB`). The theory redshift
integral is restricted to **z_max = 0.35** to match the map.


In [1]:
import os, sys
os.environ["HMFAST_COBAYA_USE_GPU"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

COBAYA_DIR = "/scratch/scratch-lxu/flamingo_data_analysis/flamingo_l2p8_lc0/catalogue_finding/cobaya_inference"
sys.path.insert(0, os.path.join(COBAYA_DIR, "code", "theory"))

import numpy as np
import healpy as hp
import pandas as pd
import pymaster as nmt
import jax.numpy as jnp
import matplotlib.pyplot as plt

# Publication-quality plot defaults
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
})


from hmfast_tsz_iter_cprofile import (
    _FID, _bin_to_18, _ELL_MIN, _ELL_MAX, _A_S_REF, _LN1E10_A_S_REF)
from hmfast.cosmology import Cosmology
from hmfast.halos import HaloModel
from hmfast.halos.mass_definition import MassDefinition
from hmfast.halos.profiles import GNFWPressureProfile
from hmfast.tracers import tSZTracer
from hmfast.tracers.tsz_completeness import (
    build_snr_grid, conditional_An_undetected, load_sigma_y0_curve)


## Configuration

In [2]:
Y_MAP = "/scratch/scratch-lxu/flamingo_repo/data/map/y_unlensed_L2p8_m9_lc0_unrotated_z0_0p35_nside4096.fits"
ITER_DATA = os.path.join(COBAYA_DIR.replace("/cobaya_inference",""), "y0_map_test", "iterative_sr", "data")

Z_MAX_MAP = 0.35                 # redshift extent of the unrotated map
B_FIXED   = 1.1093               # full-sky best-fit bias (chains/mcmc_pureGNFW_fitB)
LMAX      = 2048
MASK_RADIUS_FACTOR = 5.0
APOD_DEG  = 0.5

D3A = dict(sigma8=0.811, omega_cdm=0.118729, omega_b=0.022539,
           H0=68.1, n_s=0.967, tau_reio=0.0544)
ARN = dict(P0=8.403, c500=1.177, alpha=1.051, beta=5.4905, gamma=0.3081)

# 18 Planck-style bins: [ell_min, ell_max, ell_eff], half-open [lo, hi).
BINS = np.array([
    [   9,   12,   10.0], [  12,   16,   13.5], [  16,   21,   18.0],
    [  21,   27,   23.5], [  27,   35,   30.5], [  35,   46,   40.0],
    [  46,   60,   52.5], [  60,   78,   68.5], [  78,  102,   89.5],
    [ 102,  133,  117.0], [ 133,  173,  152.5], [ 173,  224,  198.0],
    [ 224,  292,  257.5], [ 292,  380,  335.5], [ 380,  494,  436.5],
    [ 494,  642,  567.5], [ 642,  835,  738.0], [ 835, 1085,  959.5]], dtype=float)
ELL_EFF = BINS[:, 2]

# Per q-cut: q_cat, converged iterative-SR catalogue, and the SR completeness
# triple (a_sz_sr, alpha_sr, b_sr) used by the masked hmfast theory.
CUTS = [
    dict(tag="fullsky", label="Full sky", q_cat=1.0e9, csv=None,
         a_sz_sr=-4.277981997317676, alpha_sr=0.952057528307069, b_sr=1.25),
    dict(tag="qgt5", label="q > 5", q_cat=5.0, csv="iter_05_qgt5_sample.csv",
         a_sz_sr=-4.277981997317676, alpha_sr=0.952057528307069, b_sr=1.25),
    dict(tag="qgt10", label="q > 10", q_cat=10.0, csv="iter_03_qgt10_sample.csv",
         a_sz_sr=-4.270843927746104, alpha_sr=0.9444223458089253, b_sr=1.25),
    dict(tag="qgt20", label="q > 20", q_cat=20.0, csv="iter_04_qgt20_sample.csv",
         a_sz_sr=-4.251361373150767, alpha_sr=0.9236653041902941, b_sr=1.25),
    dict(tag="qgt50", label="q > 50", q_cat=50.0, csv="iter_15_qgt50_sample.csv",
         a_sz_sr=-4.212273738811673, alpha_sr=0.8765101646932605, b_sr=1.25),
]

## Load the map and measure the full-sky D_ell^yy

`anafast` to lmax=2048, D_ell = ell(ell+1)C_ell/2pi, binned to the 18 bins by an
unweighted mean over each half-open [ell_min, ell_max) (no pixel-window or beam
correction, matching the original pipeline).

**Caching:** the measured spectra (full-sky + masked) are slow to produce
(NaMaster coupling matrices at lmax=2048). Once `data/map/nb23_datapoints/...npz`
exists we just load it and skip the map read, `anafast`, and NaMaster entirely.
Delete that file to force a fresh measurement.

In [3]:
DP_NPZ = "/scratch/scratch-lxu/flamingo_repo/data/map/nb23_datapoints/nb23_unrot_z0p35_tsz_ps.npz"
USE_CACHE = os.path.exists(DP_NPZ)

def bin_dl_from_cl(cl):
    ell = np.arange(len(cl))
    dl = ell * (ell + 1) * cl / (2.0 * np.pi)
    return np.array([dl[int(lo):int(hi)].mean() for lo, hi, _ in BINS])

if USE_CACHE:
    print("cached datapoints found -> loading measured spectra "
          "(skipping map read + anafast + NaMaster)")
else:
    ymap = hp.read_map(Y_MAP)
    nside = hp.npix2nside(ymap.size)
    print(f"map NSIDE={nside}  mean(y)={ymap.mean():.4e}  npix={ymap.size:,}")
    cl_full = hp.anafast(ymap, lmax=LMAX)
    Dl_fullsky = bin_dl_from_cl(cl_full)
    print("full-sky Dl measured (18 bins):", Dl_fullsky)

cached datapoints found -> loading measured spectra (skipping map read + anafast + NaMaster)


## Masked D_ell^yy at q_theory > {5, 10, 20, 50}

Only clusters with **z < 0.35** are masked: the rotated catalogue positions
(`theta_rot_rad`, `phi_rot_rad`) coincide with the unrotated map positions only
where the yang26 rotation is the identity, which is exactly z < 0.35. Discs of
5 R_500c, C1 0.5 deg apodisation, NaMaster decoupled pseudo-Cl with the monopole
subtracted. f_sky_eff = <w^2> gives the Knox error bars.

In [4]:
def measure_masked_dl(cut):
    df = pd.read_csv(os.path.join(ITER_DATA, cut["csv"]))
    df = df[df["z"] < Z_MAX_MAP]                      # only clusters in the map
    theta_500 = df["R_500c_Mpc"].to_numpy(float) / df["D_A_Mpc"].to_numpy(float)
    th = df["theta_rot_rad"].to_numpy(float)
    ph = df["phi_rot_rad"].to_numpy(float)
    mask = np.ones(ymap.size)
    for i in range(len(df)):
        v = hp.ang2vec(th[i], ph[i])
        pix = hp.query_disc(nside, v, MASK_RADIUS_FACTOR * theta_500[i],
                            inclusive=True, fact=4)
        mask[pix] = 0.0
    fsky_bin = float(mask.mean())
    mask_apod = nmt.mask_apodization(mask, APOD_DEG, apotype="C1")
    fsky_eff = float(np.mean(mask_apod ** 2))

    monopole = float(np.average(ymap, weights=mask_apod))
    ydelta = ymap - monopole
    ells = np.arange(LMAX + 1, dtype=int)
    bpws = np.full(LMAX + 1, -1, dtype=int)
    weights = np.zeros(LMAX + 1)
    for b, (lo, hi, _) in enumerate(BINS):
        sel = np.arange(int(lo), int(hi)); bpws[sel] = b; weights[sel] = 1.0 / len(sel)
    f_ell = ells * (ells + 1.0) / (2.0 * np.pi)
    nbin = nmt.NmtBin(bpws=bpws, ells=ells, weights=weights, lmax=LMAX, f_ell=f_ell)
    fld = nmt.NmtField(mask_apod, [ydelta], lmax=LMAX)
    wsp = nmt.NmtWorkspace(); wsp.compute_coupling_matrix(fld, fld, nbin)
    Dl_b = wsp.decouple_cell(nmt.compute_coupled_cell(fld, fld))[0]
    return np.asarray(Dl_b, float), fsky_eff, len(df), fsky_bin

if USE_CACHE:
    _d = np.load(DP_NPZ)
    measured = {c["tag"]: dict(Dl=_d[f"{c['tag']}_Dl_map"],
                               fsky_eff=float(_d[f"{c['tag']}_fsky_eff"]),
                               n=None, fsky_bin=None) for c in CUTS}
    for c in CUTS:
        print(f"{c['label']:7s}  loaded cached  f_sky_eff={measured[c['tag']]['fsky_eff']:.4f}")
else:
    measured = {"fullsky": dict(Dl=Dl_fullsky, fsky_eff=1.0, n=0, fsky_bin=1.0)}
    for cut in CUTS[1:]:
        Dl_b, fsky_eff, n, fsky_bin = measure_masked_dl(cut)
        measured[cut["tag"]] = dict(Dl=Dl_b, fsky_eff=fsky_eff, n=n, fsky_bin=fsky_bin)
        print(f"{cut['label']:7s}  masked {n:4d} clusters (z<0.35)  "
              f"f_sky_binary={fsky_bin:.4f}  f_sky_eff={fsky_eff:.4f}")

Full sky  loaded cached  f_sky_eff=1.0000
q > 5    loaded cached  f_sky_eff=0.8021
q > 10   loaded cached  f_sky_eff=0.9138
q > 20   loaded cached  f_sky_eff=0.9790
q > 50   loaded cached  f_sky_eff=0.9988


## hmfast theory at z_max = 0.35, B = 1.1093

Same `HMFastTSZIterGNFW` physics as the chains (Arnaud-2010 gNFW, sigma_lnY = 0,
D3A cosmology), but the (mass, redshift) grid is capped at z = 0.35 so the
Limber integral covers only the map's redshift range. The full-sky theory uses
q_cat -> inf (no completeness masking); each q-cut uses its SR completeness grid.
B is fixed to the full-sky best fit for every cut.

In [5]:
Z_MIN = 0.005
m = jnp.geomspace(1e10, 10**15.5, 64)
z = jnp.geomspace(Z_MIN, Z_MAX_MAP, 96)              # <-- z capped at 0.35
ell_int = jnp.geomspace(float(_ELL_MIN[0]), float(_ELL_MAX[-1]), 50)
ell_np = np.asarray(ell_int)

cosmo_seed = Cosmology(emulator_set="lcdm:v1")
hm_seed = HaloModel(cosmology=cosmo_seed.update(**_FID),
                    mass_definition=MassDefinition(500, "critical"), convert_masses=True)
prof_seed = GNFWPressureProfile(B=1.4, **ARN)
tsz_seed = tSZTracer(profile=prof_seed)
coeff, _ = load_sigma_y0_curve()

_c = cosmo_seed.update(H0=D3A["H0"], omega_cdm=D3A["omega_cdm"], omega_b=D3A["omega_b"],
                       ln1e10A_s=_LN1E10_A_S_REF, n_s=D3A["n_s"], tau_reio=D3A["tau_reio"])
ln1e10As_D3A = float(np.log(1e10 * _A_S_REF * (D3A["sigma8"] / float(np.asarray(_c.sigma8(0.0)))) ** 2))
cosmo_D3A = cosmo_seed.update(H0=D3A["H0"], omega_cdm=D3A["omega_cdm"], omega_b=D3A["omega_b"],
                              ln1e10A_s=ln1e10As_D3A, n_s=D3A["n_s"], tau_reio=D3A["tau_reio"])
hm_D3A = hm_seed.update(cosmology=cosmo_D3A)

def theory_dl(cut, B):
    snr = build_snr_grid(hm_seed, m, z, cut["a_sz_sr"], cut["alpha_sr"], cut["b_sr"], coeff=coeff)
    tsz = tsz_seed.update(profile=prof_seed.update(B=B, **ARN))
    mask1 = conditional_An_undetected(snr, sigma_lnY=0.0, q_cat=cut["q_cat"], n_power=2)
    mask2 = conditional_An_undetected(snr, sigma_lnY=0.0, q_cat=cut["q_cat"], n_power=1)
    cl1 = hm_D3A.cl_1h_masked(tsz, None, ell_int, m, z, mask1)
    cl2 = hm_D3A.cl_2h_masked(tsz, None, ell_int, m, z, mask2)
    return _bin_to_18(ell_np, np.asarray(cl1)) + _bin_to_18(ell_np, np.asarray(cl2))

theory = {}
for cut in CUTS:
    theory[cut["tag"]] = np.asarray(theory_dl(cut, B_FIXED))
    print(f"{cut['label']:7s}  theory Dl[ell~100]={theory[cut['tag']][9]:.3e}")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


I0000 00:00:1782460365.745864 2573780 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782460365.746254 2573780 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1782460366.584378 2573780 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782460366.584650 2573780 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Full sky  theory Dl[ell~100]=1.500e-13


q > 5    theory Dl[ell~100]=1.366e-14


q > 10   theory Dl[ell~100]=2.805e-14


q > 20   theory Dl[ell~100]=5.100e-14


q > 50   theory Dl[ell~100]=9.309e-14


## Goodness of fit (Knox diagonal errors)

In [6]:
from scipy.stats import chi2 as _chi2dist
def knox_err(Dl, fsky_eff):
    return np.sqrt(2.0 * Dl**2 / ((2.0*ELL_EFF + 1.0) * (BINS[:,1]-BINS[:,0]) * fsky_eff))

gof = {}
print(f"{'cut':8s} {'chi2':>9s} {'dof':>4s} {'chi2/dof':>9s} {'PTE':>10s}")
print("-"*46)
for cut in CUTS:
    t = cut["tag"]; Dl_o = measured[t]["Dl"]; Dl_t = theory[t]
    err = knox_err(Dl_o, measured[t]["fsky_eff"])
    chi2 = float(np.sum(((Dl_o - Dl_t)/err)**2)); dof = len(Dl_o)
    pte = float(_chi2dist.sf(chi2, dof))
    gof[t] = dict(err=err, chi2=chi2, pte=pte)
    print(f"{cut['label']:8s} {chi2:9.1f} {dof:4d} {chi2/dof:9.2f} {pte:10.2e}")

cut           chi2  dof  chi2/dof        PTE
----------------------------------------------
Full sky    1582.7   18     87.93   0.00e+00
q > 5       4003.4   18    222.41   0.00e+00
q > 10      3117.3   18    173.18   0.00e+00
q > 20      5020.0   18    278.89   0.00e+00
q > 50      3661.0   18    203.39   0.00e+00


## Save datapoints

Persist the measured spectra (and the matching theory / errors) as `.npy` per
cut plus one combined `.npz`, so the figure can be re-made without re-running
the SHTs or the GPU theory.

In [7]:
DP_DIR = "/scratch/scratch-lxu/flamingo_repo/data/map/nb23_datapoints"
os.makedirs(DP_DIR, exist_ok=True)
np.save(os.path.join(DP_DIR, "ell_eff.npy"), ELL_EFF)
combined = {"ell_eff": ELL_EFF, "bins": BINS}
for cut in CUTS:
    t = cut["tag"]
    # columns: ell_eff, Dl_map, sigma_knox, Dl_theory_B1.1093
    arr = np.column_stack([ELL_EFF, measured[t]["Dl"], gof[t]["err"], theory[t]])
    np.save(os.path.join(DP_DIR, f"Dl_yy_unrot_z0p35_{t}.npy"), arr)
    combined[f"{t}_Dl_map"] = measured[t]["Dl"]
    combined[f"{t}_sigma"] = gof[t]["err"]
    combined[f"{t}_Dl_theory"] = theory[t]
    combined[f"{t}_fsky_eff"] = measured[t]["fsky_eff"]
np.savez(os.path.join(DP_DIR, "nb23_unrot_z0p35_tsz_ps.npz"), **combined)
print("saved datapoints ->", DP_DIR)
for f in sorted(os.listdir(DP_DIR)):
    print("  ", f)

saved datapoints -> /scratch/scratch-lxu/flamingo_repo/data/map/nb23_datapoints
   Dl_yy_unrot_z0p35_fullsky.npy
   Dl_yy_unrot_z0p35_qgt10.npy
   Dl_yy_unrot_z0p35_qgt20.npy
   Dl_yy_unrot_z0p35_qgt5.npy
   Dl_yy_unrot_z0p35_qgt50.npy
   ell_eff.npy
   nb23_unrot_z0p35_tsz_ps.npz


## Catalogue 1-halo sum (z < 0.35)

Same Poisson sum as `14_catalogue_tsz_power_spectrum.ipynb`: per-halo GNFW form
factor with **measured** `Y_5R500c` amplitudes (`B=1` shape only), restricted to
**z < 0.35**. The dotted curves are the $\ell\to0$ white plateau
$C_\ell=\sum_i Y_{{\rm ang},i}^2/(4\pi)$ (i.e. $\sum Y_{500}^2/(4\pi D_A^4)$ in
steradians). Masked cuts sum the survivors $q<q_{\rm cut}$, matching the map
masks that remove $q>q_{\rm cut}$ clusters.

In [8]:
from scipy.integrate import quad

ARCMIN_PER_RAD = 180.0 / np.pi * 60.0
CAT = "/scratch/scratch-lxu/flamingo_repo/data/hydro_L2p8m9/catalogue/halo_catalogue_M500c_5e13_zlt3_y0q_arnaudB1.csv"

df_cat = pd.read_csv(CAT, comment="#")
df_cat = df_cat[(df_cat["Y_5R500c_Mpc2"] > 0) & (df_cat["R_500c_Mpc"] > 0) & np.isfinite(df_cat["q"])]
df_cat = df_cat[df_cat["z"] < Z_MAX_MAP]
R500 = df_cat["R_500c_Mpc"].values
Ympc2 = df_cat["Y_5R500c_Mpc2"].values
qv = df_cat["q"].values
th500 = df_cat["theta500_arcmin"].values / ARCMIN_PER_RAD
Yang = Ympc2 * (th500 / R500) ** 2                     # Y_ang = Y_5R500c / D_A^2  [sr]
print(f"catalogue clusters z<{Z_MAX_MAP}: {len(df_cat)}")
print(f"white plateau  C_white = sum(Y_ang^2)/4pi = {np.sum(Yang**2)/(4*np.pi):.3e}  [sr]")

prof_cat = GNFWPressureProfile(B=1.0)                  # fixed A10 shape, measured Y amplitude

def p_shape(x):
    cx = prof_cat.c500 * np.maximum(x, 1e-12)
    return cx ** (-prof_cat.gamma) * (1 + cx ** prof_cat.alpha) ** ((prof_cat.gamma - prof_cat.beta) / prof_cat.alpha)

I5 = quad(lambda x: p_shape(x) * x * x, 0, 5, limit=400)[0]
u_tab = np.geomspace(1e-3, 400.0, 600)
G_tab = np.array([quad(lambda x: p_shape(x) * x * x * np.sinc(u * x / np.pi), 0, 5, limit=400)[0] / I5
                  for u in u_tab])

def Ghat(u):
    return np.interp(np.clip(u, u_tab[0], u_tab[-1]), u_tab, G_tab)

def cl_1h_cat(ell_arr, mask=None):
    w2 = (Yang if mask is None else Yang[mask]) ** 2
    t = th500 if mask is None else th500[mask]
    return np.array([np.sum(w2 * Ghat(l * t) ** 2) / (4 * np.pi) for l in ell_arr])

def to_dl(ell_arr, cl):
    return ell_arr * (ell_arr + 1) * cl / (2 * np.pi)

def c_white(mask=None):
    w = Yang if mask is None else Yang[mask]
    return np.sum(w ** 2) / (4 * np.pi)

ell_fine = np.geomspace(5.0, max(ELL_EFF.max(), 1200.0), 60)
cat_sum, cat_white = {}, {}
for cut in CUTS:
    t = cut["tag"]
    m = qv < cut["q_cat"] if t != "fullsky" else None
    cat_sum[t] = to_dl(ell_fine, cl_1h_cat(ell_fine, mask=m))
    cat_white[t] = c_white(mask=m)
    if t != "fullsky":
        print(f"{cut['label']:7s}  survivors q<{cut['q_cat']:.0f}: {int((qv < cut['q_cat']).sum()):>6d}  "
              f"C_white={cat_white[t]:.3e} sr")

catalogue clusters z<0.35: 133532
white plateau  C_white = sum(Y_ang^2)/4pi = 2.906e-16  [sr]


q > 5    survivors q<5: 131560  C_white=9.767e-18 sr
q > 10   survivors q<10: 133045  C_white=6.746e-17 sr


q > 20   survivors q<20: 133441  C_white=2.267e-16 sr
q > 50   survivors q<50: 133530  C_white=2.844e-16 sr


## Figure

In [9]:
from matplotlib.lines import Line2D
colors = plt.cm.plasma(np.linspace(0.0, 0.85, len(CUTS)))
fig, (ax, axr) = plt.subplots(2, 1, figsize=(7.4, 7.2), sharex=True,
                              gridspec_kw=dict(height_ratios=[3,1], hspace=0.07))
for cut, col in zip(CUTS, colors):
    t = cut["tag"]; Dl_o = measured[t]["Dl"]; Dl_t = theory[t]; err = gof[t]["err"]
    ax.plot(ELL_EFF, Dl_t, "-", color=col, lw=2, zorder=3)
    ax.plot(ell_fine, cat_sum[t], "--", color=col, lw=1.8, zorder=2)
    ax.plot(ell_fine, to_dl(ell_fine, cat_white[t]), ":", color=col, lw=1.2, zorder=1)
    ax.errorbar(ELL_EFF, Dl_o, yerr=err, fmt="o", color=col, ms=4, capsize=2,
                elinewidth=1, mfc="white", zorder=4)
    axr.errorbar(ELL_EFF, (Dl_o - Dl_t)/err, yerr=1.0, fmt="o", color=col,
                 ms=3, capsize=2, elinewidth=1, mfc="white")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_ylabel(r"$D_\ell^{yy} = \ell(\ell+1)C_\ell^{yy}/2\pi$")
ax.set_title(r"Unrotated $z<0.35$ tSZ power spectrum: map vs hmfast ($B=1.1093$, $z_{\max}=0.35$)")
ax.grid(True, which="both", alpha=0.2)
handles = [Line2D([0],[0],color=c,marker="o",mfc="white",lw=2,label=cut["label"])
           for cut,c in zip(CUTS,colors)]
handles += [
    Line2D([0],[0],color="k",lw=2,label="hmfast theory"),
    Line2D([0],[0],color="k",ls="--",lw=1.8,label="catalogue sum (measured Y_5R500c)"),
    Line2D([0],[0],color="k",ls=":",lw=1.2,
           label=r"white plateau ($\ell\to 0$, $\sum Y_{\rm ang}^2/4\pi$)"),
    Line2D([0],[0],color="k",marker="o",mfc="white",ls="none",label="map (measured)"),
]
ax.legend(handles=handles, fontsize=7.5, loc="upper left", framealpha=0.9)
axr.axhline(0, color="k", lw=0.8)
axr.set_ylabel(r"$(D^{\rm map}-D^{\rm th})/\sigma$"); axr.set_xlabel(r"multipole $\ell$")
axr.grid(True, which="both", alpha=0.2)
OUTDIR = "/scratch/scratch-lxu/flamingo_repo/figures/nb23_unrotated_z0p35_tsz_ps"
os.makedirs(OUTDIR, exist_ok=True)
fig.savefig(os.path.join(OUTDIR, "nb23_unrotated_z0p35_tsz_ps.png"), dpi=300, bbox_inches="tight")
fig.savefig(os.path.join(OUTDIR, "nb23_unrotated_z0p35_tsz_ps.pdf"), bbox_inches="tight")
print("saved ->", OUTDIR)
plt.show()

saved -> /scratch/scratch-lxu/flamingo_repo/figures/nb23_unrotated_z0p35_tsz_ps


## Notes

The map contains only z < 0.35, so both the measurement and the theory are
restricted to that range: the theory Limber integral is capped at z_max = 0.35
and only z < 0.35 clusters are masked. B is held at the full-sky best fit
(B = 1.1093) for every cut, so the masked curves are a pure prediction, not a
refit. Error bars are the diagonal Knox approximation with f_sky_eff = <w^2>.

The dashed catalogue-sum curves and dotted white plateaus follow nb14: measured
`Y_5R500c` amplitudes, fixed A10 GNFW shape (B = 1), Poisson 1-halo sum over
z < 0.35 halos; masked cuts sum survivors with q < q_cut.